In [ ]:
-- 실행 컨텍스트 설정
USE ROLE ACCOUNTADMIN;
USE DATABASE DEMO;
USE SCHEMA ML_DEMO;

## 1. 환경 설정 (Setup)

Snowflake Feature Store 관련 라이브러리를 임포트하고 세션을 초기화합니다.

- `FeatureStore`: Feature Store 인스턴스 관리
- `FeatureView`: Feature 집합을 정의하는 뷰
- `Entity`: Feature의 기본 키(Primary Key) 정의
- `CreationMode`: Feature Store 생성 시 동작 모드 설정

## Feature Store 핵심 개념

### Feature Store란?

ML 모델에 입력되는 **피처(Feature)**를 중앙에서 정의·관리·재사용하는 저장소입니다.

**해결하는 문제:**
- 팀 간 피처 정의 불일치 → 단일 진실 원천(Single Source of Truth)
- Training-Serving Skew → 학습과 추론에서 동일한 피처 보장
- 피처 중복 가공 → 한 번 정의 후 여러 모델에서 재사용

### Snowflake 객체 매핑

| Feature Store 개념 | 실제 Snowflake 객체 | 설명 |
|---|---|---|
| Feature Store | Schema | 피처를 담는 스키마 |
| Entity | Tag | 피처의 기준 키 정의 (예: C_CUSTKEY) |
| Managed Feature View | Dynamic Table | Snowflake가 자동 갱신 |

### Managed vs External Feature View

| 구분 | Managed Feature View | External Feature View |
|---|---|---|
| 내부 구현 | Dynamic Table | View |
| 갱신 주체 | **Snowflake** (자동, `refresh_freq` 설정) | **사용자** (직접 UPDATE/INSERT, dbt 등) |
| 추가 비용 | 컴퓨팅 비용 발생 (주기적 refresh) | 없음 (View이므로) |
| 적합한 경우 | 실시간 피처 파이프라인 | 이미 가공된 테이블 등록 |


In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity
from snowflake.ml.feature_store import CreationMode
import snowflake.snowpark.functions as F
import snowflake.snowpark.types as T
from datetime import timedelta

session = get_active_session()
session.use_database("DEMO")
session.use_schema("ML_DEMO")
session.use_warehouse("COMPUTE_WH")

print(f"Connected to: {session.get_current_account()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")
print(f"Warehouse: {session.get_current_warehouse()}")

## 2. Feature Store 초기화

### Feature Store 생성 방식

| CreationMode | 설명 |
|---|---|
| `CREATE_IF_NOT_EXIST` | 없으면 생성, 있으면 기존 것 사용 (권장) |
| `FAIL_IF_NOT_EXIST` | 없으면 오류 발생 |

Snowflake Feature Store는 **Snowflake 오브젝트로 완전히 관리**됩니다:
- Feature View → Dynamic Table 또는 View로 저장
- Entity → Snowflake 메타데이터로 등록
- Dataset → Snowflake Stage에 Parquet/CSV로 저장

별도의 외부 인프라가 필요 없으며, Snowflake의 거버넌스(액세스 제어, 감사 로그)를 그대로 활용합니다.

In [ ]:
# Feature Store 초기화
# 파라미터: session, database, schema, warehouse, creation_mode
fs = FeatureStore(
    session=session,
    database="DEMO",
    name="ML_DEMO",
    default_warehouse="COMPUTE_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

print("Feature Store가 성공적으로 초기화되었습니다.")
print(f"Feature Store 위치: DEMO.ML_DEMO")

## 3. Entity 정의

**Entity**는 Feature의 기본 키(Primary Key)를 정의합니다. Feature Store에서 Feature를 조회할 때 어떤 키로 조인할지를 알려줍니다.

### TPC-H 데이터 구조에서의 Entity

```
CUSTOMER 테이블
├── C_CUSTKEY (Primary Key) ◄── Entity 키
├── C_NAME
├── C_MKTSEGMENT
└── ...
```

- Entity 이름: `CUSTOMER`
- 조인 키: `C_CUSTKEY` (고객 고유 식별자)

하나의 Entity는 여러 Feature View에서 재사용 가능합니다.

In [ ]:
# Entity 정의: 고객(Customer) Entity
customer_entity = Entity(
    name="CUSTOMER",
    join_keys=["C_CUSTKEY"]
)

# Feature Store에 Entity 등록 (이미 존재하면 기존 것 사용)
try:
    fs.register_entity(customer_entity)
except Exception:
    pass  # 이미 존재하는 경우 무시

print("Entity 'CUSTOMER' 준비 완료")
print("\n등록된 Entity 목록:")
print(fs.list_entities().to_pandas())


## 4. Managed Feature View (Dynamic Table 기반)

**Managed Feature View**는 Snowflake **Dynamic Table**로 저장됩니다:
- `refresh_freq`로 자동 갱신 주기 설정
- 소스 테이블 변경 시 자동으로 Feature 재계산
- 실시간에 가까운 Feature 최신화 가능

여기서는 ORDERS + LINEITEM + CUSTOMER를 조인하여 **모델 추론에 필요한 9개 피처**를 자동 갱신하는 Managed Feature View를 생성합니다.

### 추론 파이프라인 체인

```
ORDERS + LINEITEM + CUSTOMER (원본 변경)
    │
    ▼  [Managed FV: customer_ltv_features — 1시간마다 자동 갱신]
CUSTOMER_LTV_FEATURES (Dynamic Table — 9개 피처)
    │
    ▼  [추론 DT: LIVE_PREDICTED_LTVS — 피처 변경 시 자동 재추론]
PREDICTED_LTV (예측 결과)
```


In [ ]:
# ── 추론용 피처 계산 (ORDERS + LINEITEM + CUSTOMER 조인) ──────────────────────
# 프로덕션 추론용: 시간 필터 없이 전체 데이터 사용
# (학습 시에는 02번 노트북에서 관찰기간 필터를 적용한 별도 데이터를 사용)

orders_df = session.table("SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS")
lineitem_df = session.table("SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.LINEITEM")
customer_df = session.table("SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER")

# LINEITEM 집계 (주문 단위)
lineitem_agg = (
    lineitem_df
    .group_by("L_ORDERKEY")
    .agg(
        F.sum(F.col("L_EXTENDEDPRICE") * (F.lit(1) - F.col("L_DISCOUNT"))).alias("LINE_REVENUE"),
        F.avg("L_DISCOUNT").alias("AVG_DISCOUNT"),
        F.avg("L_QUANTITY").alias("AVG_QUANTITY"),
    )
)

# ORDERS + LINEITEM 조인
orders_with_line = (
    orders_df
    .join(lineitem_agg, orders_df["O_ORDERKEY"] == lineitem_agg["L_ORDERKEY"], how="inner")
    .select(
        orders_df["O_CUSTKEY"],
        orders_df["O_TOTALPRICE"],
        orders_df["O_ORDERDATE"],
        lineitem_agg["LINE_REVENUE"],
        lineitem_agg["AVG_DISCOUNT"],
        lineitem_agg["AVG_QUANTITY"],
    )
)

# 고객별 집계
customer_orders_agg = (
    orders_with_line
    .group_by("O_CUSTKEY")
    .agg(
        F.count("*").alias("TOTAL_ORDERS"),
        F.avg("O_TOTALPRICE").alias("AVG_ORDER_VALUE"),
        F.sum("LINE_REVENUE").alias("TOTAL_REVENUE"),
        F.avg("AVG_DISCOUNT").alias("AVG_DISCOUNT"),
        F.avg("AVG_QUANTITY").alias("AVG_QUANTITY"),
        F.max("O_ORDERDATE").alias("LAST_ORDER_DATE"),
    )
)

# CUSTOMER 조인 + DAYS_SINCE_LAST_ORDER 계산 (현재 시점 기준)
inference_features_df = (
    customer_df
    .join(customer_orders_agg, customer_df["C_CUSTKEY"] == customer_orders_agg["O_CUSTKEY"], how="inner")
    .select(
        customer_df["C_CUSTKEY"],
        customer_df["C_MKTSEGMENT"],
        customer_df["C_ACCTBAL"],
        customer_df["C_NATIONKEY"],
        customer_orders_agg["TOTAL_ORDERS"],
        customer_orders_agg["AVG_ORDER_VALUE"],
        customer_orders_agg["TOTAL_REVENUE"],
        customer_orders_agg["AVG_DISCOUNT"],
        customer_orders_agg["AVG_QUANTITY"],
        F.datediff("day", customer_orders_agg["LAST_ORDER_DATE"], F.current_date()).alias("DAYS_SINCE_LAST_ORDER"),
    )
)

print("추론용 피처 스키마 (9개 피처):")
inference_features_df.printSchema()
print(f"\\n고객 수: {inference_features_df.count():,}")
inference_features_df.show(5)


In [ ]:
# Managed Feature View 정의 (Dynamic Table 기반)
# 추론용: 전체 데이터에서 9개 피처를 자동 갱신
customer_ltv_fv = FeatureView(
    name="customer_ltv_features",
    entities=[customer_entity],
    feature_df=inference_features_df,
    refresh_freq="1 hour",  # Dynamic Table로 1시간마다 자동 갱신
    desc="추론용 고객 LTV 피처 (ORDERS+LINEITEM+CUSTOMER 조인, 9개 피처, 자동 갱신)"
)

# Feature Store에 Managed Feature View 등록 (이미 존재하면 덮어쓰기)
registered_ltv_fv = fs.register_feature_view(
    feature_view=customer_ltv_fv,
    version="1",
    overwrite=True
)

print("Managed Feature View 등록 완료")
print(f"  이름: {registered_ltv_fv.name}")
print(f"  버전: {registered_ltv_fv.version}")
print(f"  상태: {registered_ltv_fv.status}")
print(f"  갱신 주기: {registered_ltv_fv.refresh_freq}")


> ⚠️ **학습 vs 추론에서의 역할 구분**
> - **학습 시**: 고정된 학습 데이터(`CUSTOMER_FEATURES` 테이블)를 직접 사용합니다. 관찰기간이 고정되어야 Data Leakage가 없습니다.
> - **배포 후 추론 시**: Managed Feature View가 빛을 발합니다. 신규 주문이 들어오면 자동으로 피처가 갱신되어 최신 고객 행동을 반영합니다.


In [ ]:
# 등록된 모든 Feature View 목록 확인
print("Feature Store에 등록된 Feature View 목록:")
fs.list_feature_views().to_pandas()

## 5. Feature 검색 및 재사용

Feature Store의 핵심 가치는 **Feature 재사용성**입니다. 한 번 등록된 Feature는:
- 여러 ML 모델에서 동일하게 사용
- 코드 중복 없이 Feature 정의 공유
- Feature 계산 로직이 변경되면 모든 모델에 자동 반영 (Managed FV)

### Feature 조회 방법
1. `fs.list_feature_views()` - 모든 Feature View 목록
2. `fs.get_feature_view(name, version)` - 특정 Feature View 조회
3. `fs.retrieve_feature_values()` - 특정 Entity 키에 대한 Feature 값 조회

In [ ]:
# 1. 모든 Feature View 목록 조회
print("=" * 60)
print("등록된 Feature View 전체 목록")
print("=" * 60)
print(fs.list_feature_views().to_pandas().to_string(index=False))

In [ ]:
# 특정 Feature View 조회 및 Feature 컬럼 확인
print("=" * 60)
print("customer_ltv_features@1 상세 정보")
print("=" * 60)

fv = fs.get_feature_view("customer_ltv_features", "1")
print(f"이름: {fv.name}")
print(f"버전: {fv.version}")
print(f"설명: {fv.desc}")
print(f"갱신 주기: {fv.refresh_freq}")
print(f"Entity 조인 키: {[e.join_keys for e in fv.entities]}")
print("\\nFeature 컬럼 목록:")
for col in fv.feature_df.schema.fields:
    print(f"  - {col.name}: {col.datatype}")


In [ ]:
# 3. 특정 고객들에 대한 Feature 값 조회
# spine_df: 조회하려는 Entity 키 목록
# Feature Store가 자동으로 Feature View와 조인

# 조회할 고객 키 목록 (예시)
sample_customer_keys = session.create_dataframe(
    [[1], [2], [3], [4], [5]],
    schema=["C_CUSTKEY"]
)

print("특정 고객들의 Feature 값 조회:")
retrieved_features = fs.retrieve_feature_values(
    spine_df=sample_customer_keys,
    features=[fv]
)
retrieved_features.show()

### Spine DataFrame이란?

**Spine**은 "어떤 고객에 대해, 언제 시점으로 Feature를 조회할지"를 정의하는 DataFrame입니다.

```
Spine DataFrame 구조:
┌───────────┬──────────────────┬─────────────────────┐
│ C_CUSTKEY │ EVENT_TIMESTAMP  │ FUTURE_12M_REVENUE  │
│ (Entity)  │ (기준 시점, 선택) │ (레이블, 선택)       │
├───────────┼──────────────────┼─────────────────────┤
│   10001   │   1997-06-30     │      185,000        │
│   10002   │   1997-06-30     │           0         │
│   10003   │   1997-06-30     │      512,000        │
└───────────┴──────────────────┴─────────────────────┘
     ↑               ↑                   ↑
  필수 (Join 키)  선택 (PIT 기준)      선택 (레이블)
```

Feature Store는 Spine의 Entity 키를 기준으로 Feature View들을 자동 JOIN합니다:

```python
# Spine + Feature View → Training Dataset
fs.generate_training_set(
    spine_df=spine_df,          # Entity 키 + 레이블
    features=[fv1, fv2],        # 조인할 Feature View 목록
    spine_timestamp_col="ts",   # PIT 기준 시점 컬럼 (선택)
)
```

---

## 6. Training Dataset 생성

### `generate_training_set` vs `generate_dataset`

| 메서드 | 설명 | 저장 여부 |
|---|---|---|
| `generate_training_set()` | Feature를 조인한 Snowpark DataFrame 반환 | 저장 안 함 (즉시 사용) |
| `generate_dataset()` | 불변 Dataset으로 저장 (Parquet/CSV) | Snowflake Stage에 저장 |

---

### Point-in-Time Correctness (포인트-인-타임 정합성)

```
타임라인:
──────────────────────────────────────────────────────►
  t1           t2            t3              t4
(주문 발생)  (Feature 갱신)  (레이블 기준점)  (실제 학습 시점)
1992~1997.06  1997-06-30    1997-06-30       오늘(2025)
```

**각 시점의 의미 (우리 LTV 시나리오):**

| 시점 | 의미 | 우리 시나리오 |
|------|------|-------------|
| t1 | 원본 이벤트 발생 | 고객 주문 (1992 ~ 1997-06-30) |
| t2 | Feature 마지막 갱신 | 피처 계산 기준: 1997-06-30 |
| t3 | 레이블 기준 시점 | 관찰기간 종료: 1997-06-30 |
| t4 | 실제 모델 학습 날 | 오늘 (2025년) |

---

#### ❌ 잘못된 방법 — t4(오늘) 기준으로 Feature 계산

```python
# 학습을 오늘(2025년) 하면서 Feature를 "오늘" 기준으로 계산
AVG_ORDER_VALUE = orders \
    .filter(O_ORDERDATE <= "2025-01-01")   # 오늘까지 모든 주문 포함!
    .groupBy(C_CUSTKEY).avg(O_TOTALPRICE)
```

문제: 레이블은 **1997-07 ~ 1998-06 구매액**인데, Feature에 **1997-07 이후 구매 기록이 포함** → 모델이 미래 정보를 미리 알고 학습 → Data Leakage!

---

#### ✅ 올바른 방법 — t3(1997-06-30) 이전 데이터로만 Feature 계산

```python
# t3(레이블 기준점) 이전 데이터만 사용
AVG_ORDER_VALUE = orders \
    .filter(O_ORDERDATE <= "1997-06-30")   # 레이블 기준점 이전만!
    .groupBy(C_CUSTKEY).avg(O_TOTALPRICE)

# → Module 2 Step 1에서 날짜 필터를 추가한 이유가 바로 이것!
# df_orders_obs = df_orders.filter(F.col("O_ORDERDATE") <= F.lit("1997-06-30"))
```

---

#### 우리 데모에서 t2 = t3인 이유

우리 시나리오는 **모든 고객의 기준 시점이 1997-06-30으로 동일**합니다.  
→ Feature 갱신 기준(t2)과 레이블 기준점(t3)이 같아서, 날짜 필터 하나로 Leakage를 방지할 수 있습니다.

**포인트-인-타임이 복잡해지는 경우**는 고객마다 레이블 기준 시점(t3)이 다를 때입니다:

```python
# 고객마다 다른 기준 시점 (예: 월별 코호트 분석)
spine_df = session.create_dataframe([
    (고객A, datetime(2023, 3, 1)),   # 이 시점 이전 Feature만 사용
    (고객B, datetime(2023, 8, 15)),  # 이 시점 이전 Feature만 사용
    (고객C, datetime(2024, 1, 1)),   # 이 시점 이전 Feature만 사용
], schema=["C_CUSTKEY", "EVENT_TIMESTAMP"])

# generate_training_set()이 각 고객의 기준 시점에 맞는 Feature를 자동 조회
training_data = fs.generate_training_set(
    spine_df=spine_df,
    features=[fv],
    spine_timestamp_col="EVENT_TIMESTAMP",  # ← 이 컬럼 기준으로 자동 시점 필터
)
# Feature Store가 없으면 이 작업을 수동으로 처리해야 함 → 실수 위험 높음
```

→ Feature Store의 `generate_training_set(spine_timestamp_col=...)`이 각 샘플에 맞는 Feature를 **자동으로** 조회해 줍니다.

In [ ]:
# Spine DataFrame 준비: 고객 키 + FUTURE_12M_REVENUE 라벨
# ⚠️  FUTURE_12M_REVENUE를 LABEL_LTV로 rename: Feature View와 컬럼명 충돌 방지
spine_df = (
    session.table("DEMO.ML_DEMO.CUSTOMER_FEATURES")
    .select(
        F.col("C_CUSTKEY"),
        F.col("FUTURE_12M_REVENUE").alias("LABEL_LTV")  # Feature View 컬럼과 충돌 방지
    )
)

print("Spine DataFrame (Entity 키 + 라벨):")
print(f"총 레코드 수: {spine_df.count():,}")
spine_df.show(5)


In [ ]:
# Training Dataset 생성
# Feature Store가 spine_df와 Feature View를 C_CUSTKEY로 자동 조인

# 재실행 시 에러 방지
session.sql("DROP TABLE IF EXISTS DEMO.ML_DEMO.ML_DEMO_TRAINING_SET_V1").collect()
session.sql("DROP VIEW  IF EXISTS DEMO.ML_DEMO.ML_DEMO_TRAINING_SET_V1").collect()

training_df = fs.generate_training_set(
    spine_df=spine_df,
    features=[fv],  # Managed Feature View 사용
    save_as="ML_DEMO_TRAINING_SET_V1",
)

print("Training Dataset 생성 완료!")
print(f"총 레코드 수: {training_df.count():,}")
print("\\n스키마:")
training_df.printSchema()


In [ ]:
# Training Dataset 샘플 확인
print("Training Dataset 샘플 (5행):")
training_df.show(5)

# 라벨 분포 확인 (LTV 분포 확인)
print("\nFUTURE_12M_REVENUE 라벨 분포:")
training_df.group_by("FUTURE_12M_REVENUE").count().sort("FUTURE_12M_REVENUE").show()

## 7. Snowflake Dataset 버전 관리

**Snowflake Dataset**은 Feature Store의 `generate_dataset()`으로 생성되는 **불변(Immutable) 데이터셋**입니다.

### Dataset이란?

Dataset은 Snowflake의 **스키마 레벨 객체**입니다 (일반 테이블이나 Stage와는 별개).

```sql
-- SQL로도 생성/관리 가능
CREATE DATASET my_dataset;
SHOW DATASETS;
SHOW VERSIONS IN DATASET my_dataset;
```

### Dataset의 특징

| 항목 | 설명 |
|------|------|
| 객체 유형 | 스키마 레벨 객체 (`CREATE DATASET`으로 생성) |
| 불변성 | 생성 후 데이터 변경 불가 → 재현 가능한 학습 보장 |
| 버전 관리 | 하나의 Dataset에 여러 version을 저장 |
| 내부 저장 | Apache Parquet 형식으로 저장 |
| 접근 경로 | `snow://dataset/<name>/versions/<version>/` |
| 프레임워크 연동 | Snowpark, pandas, PyTorch DataPipe, TensorFlow tf.data.Dataset |
| Snowsight | Database Object Explorer에 표시되지 않음 (`SHOW DATASETS`로 확인) |

### 왜 Dataset이 필요한가?

`CUSTOMER_FEATURES` 같은 일반 테이블은 누구든 UPDATE/INSERT/DELETE 가능합니다.  
Dataset은 특정 시점의 데이터를 **영구 잠금(스냅샷)**하여 모델 재현성과 감사 추적을 보장합니다.

```
일반 테이블:  3월에 학습 → 6월에 데이터 변경 → 3월 모델 재현 불가
Dataset:     3월에 generate_dataset() → 영구 보존 → 언제든 재현 가능
```


In [ ]:
# 불변 Dataset 생성 — 학습 데이터 스냅샷을 영구 저장
# 재실행 시 에러 방지: 기존 Dataset 완전 삭제 후 재생성
try:
    session.sql("DROP DATASET IF EXISTS DEMO.ML_DEMO.customer_ltv_dataset").collect()
    print("기존 Dataset 삭제 완료")
except Exception as e:
    print(f"삭제 시도: {e}")

dataset = fs.generate_dataset(
    spine_df=spine_df,
    features=[fv],
    name="customer_ltv_dataset",
    version="v1",
    desc="고객 LTV 예측 모델 학습용 데이터셋 v1"
)

print("Dataset 생성 완료!")
print(f"Dataset 타입: {type(dataset).__name__}")
print(f"속성 목록: {[a for a in dir(dataset) if not a.startswith('_')]}")


In [ ]:
# Dataset 정보 확인
print("생성된 Dataset 정보:")
print(f"  Dataset 객체: {dataset}")

# Dataset을 DataFrame으로 읽기
print("\nDataset 데이터 확인 (5행):")
try:
    dataset_df = dataset.read.to_snowpark_dataframe()
    print(f"  레코드 수: {dataset_df.count():,}")
    dataset_df.show(5)
except Exception as e:
    print(f"  읽기 오류: {e}")
    print(f"  사용 가능한 속성: {[a for a in dir(dataset) if not a.startswith('_')]}")


In [ ]:
# 나중에 Dataset 재로드하는 방법
# Feature Store에서 이름과 버전으로 Dataset 조회
print("나중에 Dataset을 재로드하는 방법:")
print("""\n# Dataset 재로드 코드 예시:
from snowflake.ml.feature_store import FeatureStore, CreationMode

fs = FeatureStore(
    session=session,
    database='DEMO',
    name='ML_DEMO',
    default_warehouse='COMPUTE_WH',
    creation_mode=CreationMode.FAIL_IF_NOT_EXIST
)

# 등록된 Dataset 목록 확인
# 저장된 테이블을 직접 읽어서 사용
reloaded_df = session.table('DEMO.ML_DEMO.ML_DEMO_TRAINING_SET_V1')
print(f'재로드된 Dataset 레코드 수: {reloaded_df.count():,}')
""")

## 8. 정리 및 다음 단계

### 이 모듈에서 수행한 작업 요약

| 단계 | 작업 내용 | 결과물 |
|---|---|---|
| Feature Store 초기화 | `DEMO.ML_DEMO`에 Feature Store 생성 | Feature Store 인스턴스 |
| Entity 정의 | `CUSTOMER` Entity (C_CUSTKEY) 등록 | Entity 메타데이터 |
| Managed Feature View | `customer_ltv_features@1` 등록 | Dynamic Table (1시간 갱신) |
| Training Dataset | Feature Join + 라벨 결합 | `ML_DEMO_TRAINING_SET_V1` 테이블 |
| Immutable Dataset | 불변 Dataset 버전 생성 | `customer_high_value_dataset@v1` |

### 다음 단계: Module 4 - 모델 학습

```
Module 4에서는 생성된 Training Dataset을 사용하여:
├── Snowpark ML의 XGBRegressor로 모델 학습
├── 하이퍼파라미터 튜닝 (GridSearchCV)
├── 모델 성능 평가 (RMSE, MAE, R²)
└── Snowflake Model Registry에 모델 등록
```

### Feature Store 모범 사례 (Best Practices)

1. **Entity를 세분화**: 복잡한 조인 관계보다 단순한 Entity 설계 권장
2. **Managed FV는 신중하게**: 갱신 빈도가 높을수록 Warehouse 크레딧 소비 증가
3. **버전 관리**: Feature 로직 변경 시 새 버전으로 등록하여 이전 버전 유지
4. **Dataset 불변성 활용**: 재현 가능한 실험을 위해 `generate_dataset()` 사용
5. **Feature 문서화**: `desc` 파라미터로 비즈니스 의미 기록

In [ ]:
# 최종 확인: Feature Store 현황 요약
print("=" * 60)
print("Feature Store 현황 요약")
print("=" * 60)
print("\n[Entity 목록]")
print(fs.list_entities().to_pandas()[["NAME", "JOIN_KEYS"]].to_string(index=False))
print("\n[Feature View 목록]")
fv_list = fs.list_feature_views().to_pandas()
# 실제 컬럼에서 존재하는 것만 선택 (버전별 컬럼명 차이 대응)
desired = ["NAME", "VERSION", "REFRESH_FREQ", "STATUS", "STATE"]
avail = [c for c in desired if c in fv_list.columns]
print(fv_list[avail].to_string(index=False))
print("\n[Training Set 테이블]")
try:
    ts_df = session.table("DEMO.ML_DEMO.ML_DEMO_TRAINING_SET_V1")
    print(f"  ML_DEMO_TRAINING_SET_V1: {ts_df.count():,} 레코드, {len(ts_df.columns)} 컬럼")
except:
    print("  (테이블 확인 불가)")
print("\n" + "=" * 60)
print("Module 3 완료! Module 4 (모델 학습)로 진행하세요.")
print("=" * 60)